In [1]:
library(Rcpp)
library(RcppEigen)
library(RcppDist)
library(RcppArmadillo)
library(mvtnorm)
library(progress)
library(dbarts)
sourceCpp("FirstModel.cpp")

Warning message:
"package 'Rcpp' was built under R version 4.3.3"
Warning message:
"package 'RcppEigen' was built under R version 4.3.3"
Warning message:
"package 'RcppDist' was built under R version 4.3.3"
Registered S3 methods overwritten by 'RcppArmadillo':
  method               from     
  predict.fastLm       RcppEigen
  print.fastLm         RcppEigen
  summary.fastLm       RcppEigen
  print.summary.fastLm RcppEigen


Attaching package: 'RcppArmadillo'


The following objects are masked from 'package:RcppEigen':

    fastLm, fastLmPure


Warning message:
"package 'mvtnorm' was built under R version 4.3.3"
Warning message:
"package 'progress' was built under R version 4.3.3"


In [2]:
library(progress)

# DGP_2

In [3]:
#Define Helper Functions
in_cred<-function(samples, value, interval)
{
  upper_quantile<-1-(1-interval)/2
  lower_quantile<-0+(1-interval)/2

  q1<-quantile(samples, lower_quantile)
  q2<-quantile(samples, upper_quantile)

  in_cred<-ifelse(value>=q1 & value<=q2, T, F)
}

cred_width<-function(samples, interval)
{
  upper_quantile<-1-(1-interval)/2
  lower_quantile<-0+(1-interval)/2

  q1<-quantile(samples, lower_quantile)
  q2<-quantile(samples, upper_quantile)

  return(q2-q1)
}

num_gfr<-200

# Number of simulations
num_simulations <- 100

# Initialize a matrix to store the results
results_matrix <- matrix(NA, nrow = num_simulations, ncol = 30)
colnames(results_matrix) <- c("mvbcf_1k_pehe1", "mvbcf_1k_pehe2","mvbcf_0.5k_pehe1", "mvbcf_0.5k_pehe2","mvbcf_0.25k_pehe1", "mvbcf_0.25k_pehe2","mvbcf_0.1k_pehe1", "mvbcf_0.1k_pehe2",
                             "mvbcf_0.05k_pehe1", "mvbcf_0.05k_pehe2",
                             "mvbcf_1k_tau_951", "mvbcf_1k_tau_952","mvbcf_0.5k_tau_951", "mvbcf_0.5k_tau_952","mvbcf_0.25k_tau_951", "mvbcf_0.25k_tau_952","mvbcf_0.1k_tau_951", "mvbcf_0.1k_tau_952",
                             "mvbcf_0.05k_tau_951", "mvbcf_0.05k_tau_952", "mvbcf_1k_tau_951w", "mvbcf_1k_tau_952w","mvbcf_0.5k_tau_951w", "mvbcf_0.5k_tau_952w","mvbcf_0.25k_tau_951w", "mvbcf_0.25k_tau_952w","mvbcf_0.1k_tau_951w", "mvbcf_0.1k_tau_952w",
                             "mvbcf_0.05k_tau_951w", "mvbcf_0.05k_tau_952w")

# Create a progress bar
pb <- progress_bar$new(total = num_simulations)

# For loop to run the code 100 times
for (i in 1:num_simulations) {
  # Update progress bar
  pb$tick()
#Set random seed
seed_val<-i
set.seed(seed_val)

#Train Data
n<-500

X1<-runif(n)
X2<-runif(n)
X3<-runif(n)
X4<-runif(n)
X5<-runif(n)
X6<-rbinom(n, 1, 0.5)
X7<-rbinom(n, 1, 0.5)
X8<-rbinom(n, 1, 0.5)
X9<-sample(c(0, 1, 2, 3, 4), n, replace=T)
X10<-sample(c(0, 1, 2, 3, 4), n, replace=T)

X<-cbind(X1, X2, X3, X5, X6, X7, X8, X9, X10)

Mu1<-(11*sin(pi*X1*X2)+18*(X3-0.5)^2+10*X4+12*X6+X9)*10+300
Mu2<-(9*sin(pi*X1*X2)+22*(X3-0.5)^2+0*X4+8*X6+X9)*10+300

Tau1<-(2*X4+2*X5)*10
Tau2<-(1*X4+3*X5)*10

true_propensity<-X4

Z<-rbinom(n, 1, true_propensity)

Y<-cbind(Mu1+Z*Tau1, Mu2+Z*Tau2) + mvtnorm::rmvnorm(n, c(0, 0), matrix(c(50^2, 0, 0, 50^2), nrow=2, byrow=T))

#Test Data
n_test<-1000

X1_test<-runif(n_test)
X2_test<-runif(n_test)
X3_test<-runif(n_test)
X4_test<-runif(n_test)
X5_test<-runif(n_test)
X6_test<-rbinom(n_test, 1, 0.5)
X7_test<-rbinom(n_test, 1, 0.5)
X8_test<-rbinom(n_test, 1, 0.5)
X9_test<-sample(c(0, 1, 2, 3, 4), n_test, replace=T)
X10_test<-sample(c(0, 1, 2, 3, 4), n_test, replace=T)

X_test<-cbind(X1_test, X2_test, X3_test, X5_test, X6_test, X7_test, X8_test, X9_test, X10_test)

Mu1_test<-(11*sin(pi*X1_test*X2_test)+18*(X3_test-0.5)^2+10*X4_test+12*X6_test+X9_test)*10+300
Mu2_test<-(9*sin(pi*X1_test*X2_test)+22*(X3_test-0.5)^2+0*X4_test+8*X6_test+X9_test)*10+300

Tau1_test<-(2*X4_test+2*X5_test)*10
Tau2_test<-(1*X4_test+3*X5_test)*10

true_propensity_test<-X4_test

Z_test<-rbinom(n_test, 1, true_propensity_test)

Y_test<-cbind(Mu1_test+Z_test*Tau1_test, Mu2_test+Z_test*Tau2_test) + mvtnorm::rmvnorm(n_test, c(0, 0), matrix(c(50^2, 0, 0, 50^2), nrow=2, byrow=T))

#estimate of propensity score
p_mod<-bart(x.train = X, y.train = Z, x.test = X_test, k=3, verbose = FALSE)
p<-colMeans(pnorm(p_mod$yhat.train))
p_test<-colMeans(pnorm(p_mod$yhat.test))

#adding to matrix
X2<-X
X2_test<-X_test
X<-cbind(X, p)
X_test<-cbind(X_test, p_test)
Z2<-cbind(Z,Z)

#set some parameters
n_tree_mu<-50
n_tree_tau<-20
n_iter<-1000
n_burn<-500

mu_val<-1
tau_val<-0.375
v_val<-1
wish_val<-1
min_val<-1

mvbcf_1k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_1k_tau_preds1<-rowMeans(mvbcf_1k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_1k_ate1<-mean(mvbcf_1k_tau_preds1)
mvbcf_1k_tau_preds2<-rowMeans(mvbcf_1k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_1k_ate2<-mean(mvbcf_1k_tau_preds2)

mvbcf_1k_pehe1<-sqrt(mean((Tau1_test-mvbcf_1k_tau_preds1)^2))
mvbcf_1k_pehe2<-sqrt(mean((Tau2_test-mvbcf_1k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_1k_tau_951<-mean(diag(apply(mvbcf_1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_1k_tau_951w<-mean(apply(mvbcf_1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_1k_tau_952<-mean(diag(apply(mvbcf_1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_1k_tau_952w<-mean(apply(mvbcf_1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


n_iter<-500
n_burn<-250

mvbcf_0.5k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.5k_tau_preds1<-rowMeans(mvbcf_0.5k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.5k_ate1<-mean(mvbcf_0.5k_tau_preds1)
mvbcf_0.5k_tau_preds2<-rowMeans(mvbcf_0.5k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.5k_ate2<-mean(mvbcf_0.5k_tau_preds2)

mvbcf_0.5k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.5k_tau_preds1)^2))
mvbcf_0.5k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.5k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.5k_tau_951<-mean(diag(apply(mvbcf_0.5k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.5k_tau_951w<-mean(apply(mvbcf_0.5k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.5k_tau_952<-mean(diag(apply(mvbcf_0.5k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.5k_tau_952w<-mean(apply(mvbcf_0.5k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


n_iter<-250
n_burn<-125

mvbcf_0.25k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.25k_tau_preds1<-rowMeans(mvbcf_0.25k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.25k_ate1<-mean(mvbcf_0.25k_tau_preds1)
mvbcf_0.25k_tau_preds2<-rowMeans(mvbcf_0.25k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.25k_ate2<-mean(mvbcf_0.25k_tau_preds2)

mvbcf_0.25k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.25k_tau_preds1)^2))
mvbcf_0.25k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.25k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.25k_tau_951<-mean(diag(apply(mvbcf_0.25k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.25k_tau_951w<-mean(apply(mvbcf_0.25k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.25k_tau_952<-mean(diag(apply(mvbcf_0.25k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.25k_tau_952w<-mean(apply(mvbcf_0.25k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


n_iter<-100
n_burn<-50

mvbcf_0.1k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.1k_tau_preds1<-rowMeans(mvbcf_0.1k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.1k_ate1<-mean(mvbcf_0.1k_tau_preds1)
mvbcf_0.1k_tau_preds2<-rowMeans(mvbcf_0.1k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.1k_ate2<-mean(mvbcf_0.1k_tau_preds2)

mvbcf_0.1k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.1k_tau_preds1)^2))
mvbcf_0.1k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.1k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.1k_tau_951<-mean(diag(apply(mvbcf_0.1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.1k_tau_951w<-mean(apply(mvbcf_0.1k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.1k_tau_952<-mean(diag(apply(mvbcf_0.1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.1k_tau_952w<-mean(apply(mvbcf_0.1k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


n_iter<-50
n_burn<-25

mvbcf_0.05k_mod <- fast_bart(X,
                         Y,
                         Z2,
                         X2,
                         X_test, # here is the test data for the mu part of the model
                         X2_test, # here is the test data for the tau part of the model
                         0.95,
                         2,
                         0.25,
                         3,
                         diag((mu_val)^2/n_tree_mu, 2),
                         diag((tau_val)^2/n_tree_tau, 2),
                         v_val,
                         diag(wish_val, 2),
                         n_iter,
                         n_tree_mu,
                         n_tree_tau,
                         min_val,
                         num_gfr)


mvbcf_0.05k_tau_preds1<-rowMeans(mvbcf_0.05k_mod$predictions_tau_test[,1,-c(1:n_burn)])
mvbcf_0.05k_ate1<-mean(mvbcf_0.05k_tau_preds1)
mvbcf_0.05k_tau_preds2<-rowMeans(mvbcf_0.05k_mod$predictions_tau_test[,2,-c(1:n_burn)])
mvbcf_0.05k_ate2<-mean(mvbcf_0.05k_tau_preds2)

mvbcf_0.05k_pehe1<-sqrt(mean((Tau1_test-mvbcf_0.05k_tau_preds1)^2))
mvbcf_0.05k_pehe2<-sqrt(mean((Tau2_test-mvbcf_0.05k_tau_preds2)^2))

ate1 <- mean(Tau1_test)
ate2 <- mean(Tau2_test)

mvbcf_0.05k_tau_951<-mean(diag(apply(mvbcf_0.05k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, in_cred, Tau1_test, 0.95)))
mvbcf_0.05k_tau_951w<-mean(apply(mvbcf_0.05k_mod$predictions_tau_test[,1,-c(1:n_burn)], 1, cred_width, 0.95))

mvbcf_0.05k_tau_952<-mean(diag(apply(mvbcf_0.05k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, in_cred, Tau2_test, 0.95)))
mvbcf_0.05k_tau_952w<-mean(apply(mvbcf_0.05k_mod$predictions_tau_test[,2,-c(1:n_burn)], 1, cred_width, 0.95))


# Store the results in the matrix
  results_matrix[i, ] <- c(mvbcf_1k_pehe1, mvbcf_1k_pehe2, mvbcf_0.5k_pehe1, mvbcf_0.5k_pehe2, mvbcf_0.25k_pehe1, mvbcf_0.25k_pehe2, mvbcf_0.1k_pehe1, mvbcf_0.1k_pehe2,
                             mvbcf_0.05k_pehe1, mvbcf_0.05k_pehe2,
                             mvbcf_1k_tau_951, mvbcf_1k_tau_952, mvbcf_0.5k_tau_951, mvbcf_0.5k_tau_952, mvbcf_0.25k_tau_951, mvbcf_0.25k_tau_952, mvbcf_0.1k_tau_951, mvbcf_0.1k_tau_952,
                             mvbcf_0.05k_tau_951, mvbcf_0.05k_tau_952, mvbcf_1k_tau_951w, mvbcf_1k_tau_952w, mvbcf_0.5k_tau_951w, mvbcf_0.5k_tau_952w, mvbcf_0.25k_tau_951w, mvbcf_0.25k_tau_952w, mvbcf_0.1k_tau_951w, mvbcf_0.1k_tau_952w,
                             mvbcf_0.05k_tau_951w, mvbcf_0.05k_tau_952w)

}

# Export the results matrix to a CSV file
write.csv(results_matrix, "simulation_results_DGP2.csv", row.names = FALSE)

# Print a message indicating completion
cat("Simulation completed and results saved to simulation_results_DGP2.csv\n")

Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X5, X6, X7, X8, X9, X10'; match will be made by position"


Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 41145 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 24544 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 16742 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 11429 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 10828 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X5, X6, X7, X8, X9, X10'; match will be made by position"


Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 43891 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 28646 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 16894 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 12322 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 11141 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X5, X6, X7, X8, X9, X10'; match will be made by position"


...

Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 41915 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 25652 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 17808 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 13348 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 11287 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X5, X6, X7, X8, X9, X10'; match will be made by position"


Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 41703 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 25765 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 17763 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 12932 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 11358 ms


Warning message in validateXTest(test, attr(x, "term.labels"), ncol(x), colnames(x), :
"column names of 'test' does not equal that of 'x': 'X1, X2, X3, X5, X6, X7, X8, X9, X10'; match will be made by position"


Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 41675 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 25672 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 17429 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 13190 ms
Running GFR warm-start (200 iterations)...
GFR warm-start done!

Total MVBCF runtime: 11017 ms
Simulation completed and results saved to simulation_results_DGP2.csv


1m and 7.5s per iteration

1 hour and 53 minutes per 100 replications

In [4]:
print(results_matrix)

       mvbcf_1k_pehe1 mvbcf_1k_pehe2 mvbcf_0.5k_pehe1 mvbcf_0.5k_pehe2
  [1,]       37.74119      10.062373         38.19155         9.715763
  [2,]       42.07259       5.405576         42.39581         7.064330
  [3,]       36.26404       8.643772         36.16419         9.558011
  [4,]       45.78684       9.002314         44.15012         8.690295
  [5,]       35.98675       8.225859         36.69323         8.474550
  [6,]       46.42916       8.577111         47.18801         8.697186
  [7,]       39.75148       9.214479         39.22757         9.297531
  [8,]       35.87352       8.434318         36.54099         7.815605
  [9,]       37.51154       5.121568         37.69019         5.509957
 [10,]       33.16447       8.977428         34.76317         8.815892
 [11,]       36.39762      10.797731         33.80080        10.702291
 [12,]       34.78119       9.415105         34.40330         9.591586
 [13,]       31.64959      10.398604         31.51538        10.439548
 [14,]